# Bibliotecas

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

import ipywidgets as widgets # código feito por IA
from IPython.display import display, clear_output # código feito por IA

# Carregamento do JSON

In [ ]:
with open("data.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")),
     "bairro": bairro, **valores}
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])
print(df.shape)  # deve dar (6674, 7)

(6674, 7)


# Visualização dos dados

In [109]:
#Código feito com IA para auxiliar na visualização dos dados e identificação de problemas do JSON, para posteriormente podermos corrigir com código.
bairros_ordenados = sorted(df["bairro"].unique())

# Mapeamento com índice numérico (1 a 94) para facilitar a seleção
opcoes_bairros = [(f"{i+1} - {bairro}", bairro) for i, bairro in enumerate(bairros_ordenados)]

# 3. Criação dos Widgets Interativos
dropdown_bairro = widgets.Dropdown(
    options=opcoes_bairros,
    value=bairros_ordenados[0],
    description="Bairro:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

slider_niveis = widgets.IntSlider(
    value=5,
    min=1,
    max=df["nivel_m"].nunique(),
    step=1,
    description="Qtd. Níveis:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

output_area = widgets.Output()

# 4. Função de atualização dinâmica dos resultados
def atualizar_exibicao(change=None):
    bairro_selecionado = dropdown_bairro.value
    qtd_niveis = slider_niveis.value
    
    # Filtrar o DataFrame pelo bairro selecionado e ordenar pelo nível do rio (m)
    df_bairro = df[df["bairro"] == bairro_selecionado].sort_values(by="nivel_m")
    resultado = df_bairro.head(qtd_niveis)
    
    with output_area:
        clear_output(wait=True)
        print(f"Bairro selecionado: {bairro_selecionado}")
        print(f"Exibindo os primeiros {len(resultado)} níveis de inundação:")
        display(resultado)

# Vincular os eventos dos widgets à função de atualização
dropdown_bairro.observe(atualizar_exibicao, names='value')
slider_niveis.observe(atualizar_exibicao, names='value')

# 5. Renderizar a interface interativa no Notebook
interface = widgets.VBox([
    widgets.HTML(value="### Painel Interativo de Análise de Inundação"),
    dropdown_bairro,
    slider_niveis,
    widgets.HTML(value="<hr/>"),
    output_area
])

display(interface)
atualizar_exibicao()

In [88]:
df.info()# Sem valores nulos, mas pode haver valores com 0, representando um dado ausente!

#Verificando se tem algum bairro com total_buildings ou population igual a 0.
print(len(df[df["total_buildings"]== 0]))
print(len(df[df["population"] == 0]))

<class 'pandas.DataFrame'>
RangeIndex: 6674 entries, 0 to 6673
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   nivel_m              6674 non-null   float64
 1   bairro               6674 non-null   str    
 2   affected_buildings   6674 non-null   int64  
 3   total_buildings      6674 non-null   int64  
 4   percentage           6674 non-null   int64  
 5   population           6674 non-null   int64  
 6   affected_population  6674 non-null   int64  
dtypes: float64(1), int64(5), str(1)
memory usage: 365.1 KB
0
0


In [90]:
"""
Essa célula é responsável por verifificar se todos os níveis possuem os 94 bairros cadastrados 
"""
bairros_unicos = df["bairro"].nunique()
niveis_unicos = df["nivel_m"].nunique()
esperado_linhas = bairros_unicos * niveis_unicos

print(f"Total de bairros: {bairros_unicos}")
print(f"Total de níveis de água únicos: {niveis_unicos}")
print(f"Total de registros encontrados: {esperado_linhas}")
print(df.shape)  # deve dar (6674, 7)


Total de bairros: 94
Total de níveis de água únicos: 71
Total de registros encontrados: 6674
(6674, 7)


In [47]:
inconsistencias_totais = []

bairros_unicos = df["bairro"].unique()

#Passa por cada bairro um por um
for bairro in bairros_unicos:
    
    # Filtra apenas os dados do bairro atual
    df_bairro = df[df["bairro"] == bairro]
    
    # Ordena do menor nível para o maior nível do rio
    df_bairro = df_bairro.sort_values("nivel_m")
    
    # Transforma as colunas em listas do Python para facilitar a leitura
    niveis = df_bairro["nivel_m"].tolist()
    predios = df_bairro["affected_buildings"].tolist()
    
    # Compara cada nível com o nível anterior
    for i in range(1, len(predios)):
        predios_atual = predios[i]
        predios_anterior = predios[i - 1]
        
        # Se o número de prédios afetados diminuiu com o rio subindo, temos um erro
        if predios_atual < predios_anterior:
            inconsistencias_totais.append({
                "bairro": bairro,
                "nivel_m": niveis[i],
                "affected_buildings": predios_atual,
                "predios_nivel_anterior": predios_anterior
            })

# Transformamos o resultado final em um DataFrame
df_inconsistencias_fisicas = pd.DataFrame(inconsistencias_totais)

# Mostramos o resultado na tela
print(f"Total de inconsistências encontradas: {len(df_inconsistencias_fisicas)}")

if len(df_inconsistencias_fisicas) > 0:
    display(df_inconsistencias_fisicas.head(len(df_inconsistencias_fisicas)))
else:
    print("Nenhuma inconsistência encontrada.")

Total de inconsistências encontradas: 12


,bairro,nivel_m,affected_buildings,predios_nivel_anterior
0,Arquipélago,3.1,4680,4685
1,Arquipélago,3.6,4680,4685
2,Arquipélago,4.1,4674,4685
3,Arquipélago,4.6,4680,4685
4,Arquipélago,5.1,4680,4685
5,Arquipélago,5.4,4680,4688
6,Centro Histórico,9.7,0,2050
7,Espírito Santo,3.4,633,637
8,Espírito Santo,3.5,25,633
9,Espírito Santo,4.8,47,2101


In [44]:
#Correção das inconsistencias
df_limpo = df.copy() #Cópia do DataFrame original

#ordena os bairros por ordem alfabética, e os nívels do menor para o maior(3.0m até 10.0m), reset_index(drop=True) refaz a numeação da linhas das tabelas(0,1,2,...)
df_limpo = df_limpo.sort_values(by=["bairro", "nivel_m"]).reset_index(drop=True)

#Corrige o múmero de prédios afetados caso ele caia após o nível subir. O valor se torna o maior valor encontrado até o momento
df_limpo["affected_buildings"] = df_limpo.groupby("bairro")["affected_buildings"].cummax()#Cumulative Maximum/Máximo Acumulado.


# corrige os campos dependentes da coluna affected_buildings (percentage e affected_population) 
df_limpo["percentage"] = (df_limpo["affected_buildings"] / df_limpo["total_buildings"]) * 100
df_limpo["affected_population"] = df_limpo["population"] * df_limpo["percentage"] / 100

#apenas para verificar
print(df_limpo.shape)

#Exibição das inconsistências corrigidas

#transformamos em uma lista de tuplas (bairro, nível) onde ocorreu a inconsistência
inconsistencias_fisicas = list(zip(df_inconsistencias_fisicas["bairro"], df_inconsistencias_fisicas["nivel_m"])) 
# verifica se cada tupla está no DataFrame, nas linhas que encontrar marca como True
condicao_filtro = df_limpo.set_index(["bairro", "nivel_m"]).index.isin(inconsistencias_fisicas) 
df_conferencia_fisica = df_limpo[condicao_filtro] # mantém apenas onde as linhas deram True

print(f"\n Verificação dos {len(df_conferencia_fisica)} bairros que apresentavam inconsistências, agora corrigidos:")
display(df_conferencia_fisica[["bairro", "nivel_m", "affected_buildings"]])



(6674, 7)

 Verificação dos 12 bairros que apresentavam inconsistências, agora corrigidos:


,bairro,nivel_m,affected_buildings
214,Arquipélago,3.1,4685
219,Arquipélago,3.6,4685
224,Arquipélago,4.1,4685
229,Arquipélago,4.6,4685
234,Arquipélago,5.1,4685
237,Arquipélago,5.4,4688
1274,Centro Histórico,9.7,2050
1779,Espírito Santo,3.4,637
1780,Espírito Santo,3.5,637
1793,Espírito Santo,4.8,2101


In [31]:

df_copia = df.copy()

# Recalcula a porcentagem e a população afetada baseada nas fórmulas do documento
df_copia["calc_percentage"] = (df_copia["affected_buildings"] / df_copia["total_buildings"]) * 100
df_copia["calc_pop_affected"] = df_copia["population"] * df_copia["percentage"] / 100

# Verifica onde a diferença entre o dado original e o calculado é maior que uma margem pequena de arredondamento
erro_perc = (df_copia["percentage"] - df_copia["calc_percentage"]).abs() > 0.01
erros_encontrados = df_copia[erro_perc]

print(f"Total de registros com divergências")
if len(erros_encontrados) > 0:
    display(erros_encontrados.head(5))
else:
    print(" Todos os cálculos de porcentagem estão consistentes")

Total de registros com divergências


,nivel_m,bairro,affected_buildings,total_buildings,percentage,population,affected_population,calc_percentage,calc_pop_affected
2,3.0,Anchieta,28,589,5,791,40,4.753820,39.55
3,3.0,Arquipélago,4685,4688,100,6411,6411,99.936007,6411.00
7,3.0,Belém Novo,95,6012,2,9851,198,1.580173,197.02
10,3.0,Boa Vista do Sul,306,2450,13,2703,352,12.489796,351.39
17,3.0,Centro Histórico,42,4312,1,30569,306,0.974026,305.69


In [34]:

# Recalcula a porcentagem de forma precisa com base nos prédios afetados e totais
df_limpo["percentage"] = (df_limpo["affected_buildings"] / df_limpo["total_buildings"]) * 100

# Recalcula a população afetada utilizando a porcentagem precisa
df_limpo["affected_population"] = df_limpo["population"] * df_limpo["percentage"] / 100

print(" Correção dos cálculos inseridas no df_limpo")
print(df_limpo.shape)

# Exibe as primeiras linhas para conferir como ficou
display(df_limpo.head(5))

 Correção dos cálculos inseridas no df_limpo
(6674, 7)


,nivel_m,bairro,affected_buildings,total_buildings,percentage,population,affected_population
0,3.0,Aberta dos Morros,0,4696,0.000000,9854,0.000000
1,3.1,Aberta dos Morros,0,4696,0.000000,9854,0.000000
2,3.2,Aberta dos Morros,0,4696,0.000000,9854,0.000000
3,3.3,Aberta dos Morros,0,4696,0.000000,9854,0.000000
4,3.4,Aberta dos Morros,1,4696,0.021295,9854,2.098382


In [38]:
# 1. Recalcula a porcentagem e a população afetada de forma precisa no df_limpo
df_limpo["percentage"] = (df_limpo["affected_buildings"] / df_limpo["total_buildings"]) * 100
df_limpo["affected_population"] = df_limpo["population"] * df_limpo["percentage"] / 100

print("Correção dos cálculos inserida e aplicada no df_limpo")
print(f"Dimensões do DataFrame limpo: {df_limpo.shape}\n")

# 2. Pega automaticamente a lista de bairros que apresentavam divergência na célula anterior
bairros_com_erro = erros_encontrados["bairro"].tolist()

# 3. Filtra o df_limpo para mostrar exatamente esses bairros no nível 3.0m já com os valores corrigidos
df_conferencia = df_limpo[
    (df_limpo["nivel_m"] == 3.0) & 
    (df_limpo["bairro"].isin(bairros_com_erro))
]

print(f" Conferência automática dos {len(bairros_com_erro)} bairros corrigidos no df_limpo (Nível 3.0m):")
display(df_conferencia[["nivel_m", "bairro", "affected_buildings", "total_buildings", "percentage", "affected_population"]])

Correção dos cálculos inserida e aplicada no df_limpo
Dimensões do DataFrame limpo: (6674, 7)

 Conferência automática dos 2620 bairros corrigidos no df_limpo (Nível 3.0m):


,nivel_m,bairro,affected_buildings,total_buildings,percentage,affected_population
0,3.0,Aberta dos Morros,0,4696,0.000000,0.000000
142,3.0,Anchieta,28,589,4.753820,37.602716
213,3.0,Arquipélago,4685,4688,99.936007,6406.897398
284,3.0,Auxiliadora,0,1995,0.000000,0.000000
355,3.0,Azenha,0,3317,0.000000,0.000000
497,3.0,Belém Novo,95,6012,1.580173,155.662841
710,3.0,Boa Vista do Sul,306,2450,12.489796,337.599184
923,3.0,Camaquã,0,7897,0.000000,0.000000
1136,3.0,Cavalhada,0,9111,0.000000,0.000000
1207,3.0,Centro Histórico,42,4312,0.974026,297.750000
